# IaC in real DevOps projects — Terraform module exploration

> last_verified: 2026-08-12 · n/a (concept)

This notebook layers the Infrastructure as Code pattern (declarative, version-controlled infrastructure) onto the module-reuse idea borrowed from software libraries: a Terraform module is a folder of `.tf` files that packages related resources into one callable unit. Reading it is like reading an API contract — inputs in, outputs out, resources in between — before wiring it into a real project.

## What a Terraform module is

A module is a folder of `.tf` files that bundles related resources into one callable unit. The **root module** is what `terraform apply` runs directly; a **child module** is referenced from a root by a `module` block. Each module exposes an interface:

- `variables.tf` — the inputs a caller must (or may) provide.
- `outputs.tf` — the values a caller can read back after apply.
- `main.tf` (or any `.tf`) — the resources and data sources themselves.

Reading a module is the same as reading an API contract: inputs in, outputs out, resources in between.

In [ ]:
%%bash

# Explore a module the way I'd read an unfamiliar library:
# 1) inventory the .tf files, 2) read the interface, 3) read one resource.

MODULE_DIR=$(mktemp -d)

# A minimal child module: provisions a single placeholder resource.
cat > "$MODULE_DIR/variables.tf" <<'TF'
variable "name" {
  type        = string
  description = "Name of the resource"
}

variable "tags" {
  type    = map(string)
  default = {}
}
TF

cat > "$MODULE_DIR/main.tf" <<'TF'
resource "null_resource" "placeholder" {
  triggers = {
    name = var.name
  }
}
TF

cat > "$MODULE_DIR/outputs.tf" <<'TF'
output "name" {
  value = var.name
}
TF

echo "=== Module file inventory ==="
ls "$MODULE_DIR"

echo "\n=== Interface: variables.tf ==="
grep -E 'variable "|default' "$MODULE_DIR/variables.tf"

echo "\n=== Interface: outputs.tf ==="
grep -E 'output "' "$MODULE_DIR/outputs.tf"

rm -rf "$MODULE_DIR"

## Calling a module from a root module

Once the interface is clear, the root module calls the child with a `module` block. The `source` points at a local path or a registry reference, and `version` pins it when it comes from a public registry. Every required variable with no default must be passed, or the plan fails at validate time — that is the fastest way to catch a wrong interface read.

In [ ]:
%%bash

# Demonstrate the caller side: a root module consuming the child module above.

ROOT_DIR=$(mktemp -d)
mkdir -p "$ROOT_DIR/modules/placeholder"

cat > "$ROOT_DIR/modules/placeholder/main.tf" <<'TF'
resource "null_resource" "placeholder" {
  triggers = {
    name = var.name
  }
}
TF

cat > "$ROOT_DIR/modules/placeholder/variables.tf" <<'TF'
variable "name" {
  type = string
}
TF

cat > "$ROOT_DIR/main.tf" <<'TF'
module "placeholder" {
  source = "./modules/placeholder"
  name   = var.service
}
TF

cat > "$ROOT_DIR/variables.tf" <<'TF'
variable "service" {
  type    = string
  default = "web"
}
TF

echo "=== Is 'main' (the first resource block) callable? ==="
grep -n 'module "' "$ROOT_DIR/main.tf"

echo "\n=== Where does the module source point? ==="
grep -E 'source' "$ROOT_DIR/main.tf"

echo "\n=== Required input the caller must satisfy ==="
grep -E 'variable "' "$ROOT_DIR/modules/placeholder/variables.tf"

rm -rf "$ROOT_DIR"

## Reading a real-world module

Production modules follow the same shape but are bigger. When I explore one in the wild, this checklist tells me what I'm looking at:

1. **Variables first** — which are required (no default), which are optional, and what the defaults assume.
2. **Outputs after** — what the module hands back (IDs, ARNs, names) so the root can pass them between modules.
3. **Resources oldest** — the meat; count them to estimate blast radius.
4. **Version pinning** — registry-sourced modules should be pinned, never floating; a silently-changing module defeats reproducibility.

This exploration loop — inventory, read interface, read resources, then call — is the same for a 20-line module and a published cloud networking module. The interface discipline is what makes the pattern scale across an organization.